# Rerun with 128 hidden units, batch size 16 and a 200-epoch RL cap

Runs all three recurrent models, four horizons and seeds 41–43 on GPU.
The setup cell applies the three confirmed settings before generating seed configs,
even if the cloned branch still contains the previous defaults.

**RL uses a 200-epoch maximum with patience 5**, so it may stop earlier.
TL retains the currently configured 10+10 caps and patience policy. Learning rates,
weight decay and other pipeline choices have not been verified against the old MacBook run;
this notebook restores the three settings you confirmed, not an exact historical execution.

Outputs go to **MyDrive/bg-results-h128-b16-e200**, with a separate local checkout,
so previous results are preserved and do not cause these fits to be skipped.
Select an A100 runtime and run the cells in order. Existing completed runs in this
new namespace can be resumed using the notebook's usual workflow.


## 1 · Mount Drive and set paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# --- edit these to match your Drive ---------------------------------------
DRIVE_DATA    = "/content/drive/MyDrive/ohiot1dm"      # must contain 2018/ and 2020/
DRIVE_RESULTS = "/content/drive/MyDrive/bg-results-h128-b16-e200"    # results are mirrored here
REPO_DIR      = "/content/BG-forecasting-h128-b16-e200"

# Where the code comes from.
#   "git"   -> clone REPO_URL at BRANCH
#   "drive" -> copy DRIVE_REPO (use this if the branch is not pushed)
SOURCE     = "git"
REPO_URL   = "https://github.com/beatriz-fulgencio/BG-forecasting.git"
BRANCH     = "bench2"
DRIVE_REPO = "/content/drive/MyDrive/BG-forecasting"

SEEDS    = [41, 42, 43]
MODELS   = ["gru", "lstm", "rnn"]
HORIZONS = [15, 30, 45, 60]
# ---------------------------------------------------------------------------

import os, pathlib
for d in (DRIVE_RESULTS, f"{DRIVE_RESULTS}/experiments", f"{DRIVE_RESULTS}/logs"):
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)
print("Drive ready.")

## 2 · Get the code

`run_seed_major.sh`, `merge_seed_runs.py` and the F=4 `configs/full_*.yaml` must
be present. Those twelve configs are the run: `run_seed_major.sh` derives each
single-seed config in `configs/seedwise/` straight from them, changing only the
name and the seed list. So the check below reads their contents, not just their
names — a stale checkout has the file names too. If the corrected configs are
not committed and pushed, set `SOURCE = "drive"` above and put a copy of the
repo folder in Drive instead.

In [ ]:
import shutil, subprocess, sys, pathlib, yaml

if SOURCE == "git":
    if not pathlib.Path(REPO_DIR).is_dir():
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
    else:
        pull = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"],
                              capture_output=True, text=True)
        if pull.returncode != 0:
            print("WARNING: git pull --ff-only failed, so this checkout may be stale "
                  "and the config check below is what stands between you and a "
                  "wrong-input run.\n" + (pull.stderr or pull.stdout).strip() + "\n")
elif SOURCE == "drive":
    if not pathlib.Path(REPO_DIR).is_dir():
        shutil.copytree(DRIVE_REPO, REPO_DIR, ignore=shutil.ignore_patterns("results", "output", "__pycache__"))
else:
    raise ValueError("SOURCE must be 'git' or 'drive'")

os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)

# The twelve full-run configs are what gets trained: run_seed_major.sh derives
# configs/seedwise/full_<model>_<horizon>min_seed<N>.yaml from them, rewriting
# only the experiment name, the description and the seed list.
CONFIGS = [f"configs/full_{m}_{h}min.yaml" for m in MODELS for h in HORIZONS]

missing = [f for f in ["run_seed_major.sh", "merge_seed_runs.py"] + CONFIGS
           if not pathlib.Path(f).is_file()]
if missing:
    raise SystemExit(
        "Missing from this checkout:\n  " + "\n  ".join(missing) +
        "\n\nCommit and push them, or set SOURCE = 'drive'."
    )

# Apply confirmed original settings even if the remote branch still has the newer defaults.
# All unlisted hyperparameters stay as configured; this is not a verified restoration
# of unspecified old learning rates or weight decay.
from benchmark.configs import load_config
for cfg_path in CONFIGS:
    path = pathlib.Path(cfg_path)
    cfg = yaml.safe_load(path.read_text())
    cfg['model']['architecture']['hidden_size'] = 128
    cfg['training'].update(batch_size=16, epochs=200, device='cuda', seeds=SEEDS)
    path.write_text(yaml.safe_dump(cfg, sort_keys=False))
    resolved = load_config(path)
    assert (resolved.model.architecture['hidden_size'], resolved.training.batch_size,
            resolved.training.epochs) == (128, 16, 200)
    print(cfg_path, 'hidden=128 batch=16 RL cap=200 patience=',
          resolved.training.early_stopping_patience)

# Protect this results namespace from later configuration changes.
import json, hashlib
manifest = {p: yaml.safe_load(pathlib.Path(p).read_text()) for p in CONFIGS}
guard = pathlib.Path(DRIVE_RESULTS) / 'rerun_configs.json'
if guard.exists() and json.loads(guard.read_text()) != manifest:
    raise RuntimeError('This Drive folder belongs to different configs. Choose a fresh DRIVE_RESULTS and REPO_DIR.')
guard.write_text(json.dumps(manifest, indent=2, sort_keys=True))

# Same two flags run_seed_major.sh preflights, checked here so a stale clone
# fails now with the offending file named, rather than after the dataset is
# staged. unimodal must be false (glucose + basal + bolus + carbs), and feature
# engineering off or the cyclical hour columns push F=4 to F=6.
wrong = []
for cfg_path in CONFIGS:
    pre = yaml.safe_load(pathlib.Path(cfg_path).read_text())["preprocessing"]
    if pre.get("unimodal") is not False:
        wrong.append(f"{cfg_path}: unimodal is {pre.get('unimodal')!r}, not false "
                     f"-> would train on glucose only (F=1)")
    if pre.get("include_feature_engineering") is not False:
        wrong.append(f"{cfg_path}: include_feature_engineering is "
                     f"{pre.get('include_feature_engineering')!r}, not false -> F=6, not 4")
if wrong:
    raise SystemExit(
        "These full-run configs would not train on F=4:\n  " + "\n  ".join(wrong) +
        f"\n\nFix them on {BRANCH} and push, or set SOURCE = 'drive'."
    )

os.chmod("run_seed_major.sh", 0o755)
head = subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--format=%h %cd %s",
                       "--date=short"], capture_output=True, text=True).stdout.strip()
print(f"Repo ready at {REPO_DIR}" + (f"\n  commit: {head}" if head else ""))
print(f"  {len(CONFIGS)} full-run config(s) present, all F=4.")

# The configs ask for cuda. Say now whether this runtime can give it, rather
# than letting the first cell fail an hour in.
import torch
if torch.cuda.is_available():
    print(f"  device: cuda -> {torch.cuda.get_device_name(0)}")
else:
    print("  WARNING: torch reports no CUDA device. The configs set device: cuda,\n"
          "           so every cell will fail. Runtime > Change runtime type > T4 GPU,\n"
          "           or set device: cpu in configs/full_*.yaml.")

## 3 · Stage the OhioT1DM data

In [ ]:
import shutil, pathlib

# 6 patients x 2 releases x train+test = 24 XML files.
COHORT = {"2018": [559, 563, 570, 575, 588, 591],
          "2020": [540, 544, 552, 567, 584, 596]}

dst = pathlib.Path(REPO_DIR) / "data" / "raw" / "ohiot1dm"
src = pathlib.Path(DRIVE_DATA)
if not src.is_dir():
    raise SystemExit(f"{DRIVE_DATA} not found. Upload your OhioT1DM copy there first.")

# Copy file by file. Skipping a release whose directory already exists leaves an
# interrupted copy permanently incomplete -- the run then fails on the handful of
# files that never arrived, with everything else in place.
copied = 0
for release, patients in COHORT.items():
    for mode, suffix in (("train", "training"), ("test", "testing")):
        (dst / release / mode).mkdir(parents=True, exist_ok=True)
        for pid in patients:
            name = f"{pid}-ws-{suffix}.xml"
            target, source = dst / release / mode / name, src / release / mode / name
            if target.is_file() or not source.is_file():
                continue          # already staged, or absent from Drive (reported below)
            shutil.copy2(source, target)
            copied += 1
print(f"{copied} file(s) copied from Drive.")

# These are the exact paths the runner checks in validate_data_files
# (benchmark/configs/config_manager.py). Fail here rather than 36 cells later,
# and say for each one whether Drive has it at all.
missing = []
for release, patients in COHORT.items():
    for mode, suffix in (("train", "training"), ("test", "testing")):
        for pid in patients:
            name = f"{pid}-ws-{suffix}.xml"
            if not (dst / release / mode / name).is_file():
                in_drive = (src / release / mode / name).is_file()
                missing.append(
                    f"{release}/{mode}/{name}" +
                    ("  (in Drive, but the copy failed)" if in_drive
                     else f"  (NOT in {DRIVE_DATA}/{release}/{mode}/)"))

staged = sorted(dst.glob("*/*/*.xml"))
print(f"{len(staged)}/24 XML file(s) staged under {dst}")
if missing:
    raise SystemExit(
        f"{len(missing)} OhioT1DM file(s) missing:\n  " + "\n  ".join(missing) +
        "\n\nUpload them to Drive, then re-run this cell -- it fills in whatever "
        "is absent rather than starting over."
    )
print("All 24 files present.")

## 4 · Restore earlier progress, then check the plan

Pulls anything already finished back from Drive so finished cells are skipped,
then prints the plan without running anything.

In [ ]:
import pathlib, shutil, subprocess

EXP_DIR = pathlib.Path(REPO_DIR) / "results" / "experiments"
EXP_DIR.mkdir(parents=True, exist_ok=True)

restored = 0
for d in sorted(pathlib.Path(f"{DRIVE_RESULTS}/experiments").glob("experiment_*")):
    target = EXP_DIR / d.name
    if not target.exists():
        shutil.copytree(d, target)
        restored += 1
print(f"Restored {restored} experiment dir(s) from Drive.\n")

env = dict(os.environ, DRY_RUN="1",
           SEEDS=" ".join(map(str, SEEDS)),
           MODELS=" ".join(MODELS),
           HORIZONS=" ".join(map(str, HORIZONS)),
           PYTHON=sys.executable)
print(subprocess.run(["./run_seed_major.sh"], cwd=REPO_DIR, env=env,
                     capture_output=True, text=True).stdout)

## 5 · Confirm the inputs really are F=4

Builds one patient's dataset — preprocessing only, no training — with the
window, horizon and feature flags **read out of the full-run config** the
seedwise run derives from, then asserts the four feature columns. Cheap
insurance against discovering at analysis time that the runs were glucose-only.

In [ ]:
from benchmark.data.loaders import load_ohiot1dm_data
from benchmark.data.preprocessors import preprocess_ohiot1dm_data
from benchmark.data.torch_dataset import prepare_patient_datasets
import io, contextlib, pathlib, yaml

# Read the parameters from the config that will actually be trained rather than
# repeating its values here, so this cell cannot agree with itself while
# disagreeing with the run.
CHECK_CONFIG = f"configs/full_{MODELS[0]}_{HORIZONS[0]}min.yaml"
cfg = yaml.safe_load((pathlib.Path(REPO_DIR) / CHECK_CONFIG).read_text())
pre, dat = cfg["preprocessing"], cfg["data"]
print(f"{CHECK_CONFIG}: window_size={pre['window_size']} "
      f"prediction_horizon={pre['prediction_horizon']} "
      f"sampling_rate={pre['sampling_rate']} "
      f"unimodal={pre['unimodal']} "
      f"include_feature_engineering={pre['include_feature_engineering']}")

pid = 559
with contextlib.redirect_stdout(io.StringIO()):          # preprocessing is chatty
    tr = preprocess_ohiot1dm_data(
        load_ohiot1dm_data(dat["root"], patient_ids=[pid], mode="train",
                           version="2018", sampling_rate=pre["sampling_rate"]),
        include_feature_engineering=pre["include_feature_engineering"])
    te = preprocess_ohiot1dm_data(
        load_ohiot1dm_data(dat["root"], patient_ids=[pid], mode="test",
                           version="2018", sampling_rate=pre["sampling_rate"]),
        include_feature_engineering=pre["include_feature_engineering"])
    train_ds, test_ds = prepare_patient_datasets(
        tr[pid], te[pid], pre["window_size"], pre["prediction_horizon"],
        unimodal=pre["unimodal"])

print("features:", train_ds.feature_columns)
print("train:", train_ds.data.shape, " test:", test_ds.data.shape)
assert train_ds.feature_columns == ["glucose", "basal", "bolus", "carbs"], train_ds.feature_columns
assert train_ds.data.shape[1] == 4
print("\nF=4 confirmed.")

## 6 · Run

One cell at a time, seed-major, mirroring to Drive after each. Re-running this
cell after a disconnect picks up where it stopped. Full logs go to Drive;
only progress lines are printed here.

In [ ]:
import subprocess, shutil, pathlib, time, sys, os, collections

def mirror_to_drive():
    """Copy completed experiment dirs to Drive. Incomplete runs are skipped."""
    dst = pathlib.Path(f"{DRIVE_RESULTS}/experiments")
    copied = 0
    for d in EXP_DIR.glob("experiment_*"):
        if not (d / "aggregate_metrics.json").is_file():
            continue
        target = dst / d.name
        if target.exists():
            continue
        shutil.copytree(d, target)
        copied += 1
    return copied

KEEP = ("[", "seed ", "  ok ", "  FAILED", "Preflight", "Nothing", "Done:", "config:", "ERROR", "Traceback")

def run_cell(seed, model, horizon):
    name = f"full_{model}_{horizon}min_seed{seed}"
    env = dict(os.environ, SEEDS=str(seed), MODELS=model, HORIZONS=str(horizon),
               PYTHON=sys.executable, LOG_DIR=f"{DRIVE_RESULTS}/logs")
    t0 = time.time()
    proc = subprocess.Popen(["./run_seed_major.sh"], cwd=REPO_DIR, env=env,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    tail = collections.deque(maxlen=40)   # keep the end of the output for failures
    for line in proc.stdout:
        tail.append(line.rstrip())
        if any(k in line for k in KEEP):
            print(line.rstrip(), flush=True)
    rc = proc.wait()
    if rc != 0:
        # KEEP hides everything that is not a progress line, which on a failure
        # is exactly the part you need. Print the end of the output verbatim.
        print(f"    --- last {len(tail)} line(s) of {name} ---", flush=True)
        for line in tail:
            print("    " + line, flush=True)
        print(f"    --- full log: {DRIVE_RESULTS}/logs/{name}.log ---", flush=True)
    copied = mirror_to_drive()
    print(f"    {'OK ' if rc == 0 else 'FAIL'} {name} in {(time.time()-t0)/60:.0f} min"
          f"  (mirrored {copied} dir(s) to Drive)", flush=True)
    return rc

failures = []
for seed in SEEDS:                      # seed-major: the outer loop is the seed
    print(f"\n{'='*62}\n SEED {seed}\n{'='*62}", flush=True)
    for model in MODELS:
        for horizon in HORIZONS:
            if run_cell(seed, model, horizon) != 0:
                failures.append(f"full_{model}_{horizon}min_seed{seed}")

print("\nFailed cells:", failures if failures else "none")

[link text](https://)## 7 · Merge the three seeds per cell

Each single-seed run lands in its own parent directory, whose aggregate reports
null for every dispersion field — one seed has no spread. The downstream analysis
expects one parent per cell holding all three seeds, with cross-seed means and
Student-t intervals. This rebuilds that using the benchmark's own aggregation
code, so the merged numbers come from the same code path a three-seed run uses.

Run it once all three seeds of a cell are done; cells that aren't complete are
reported and skipped.

In [ ]:
import subprocess, sys, shutil, pathlib

args = [sys.executable, "merge_seed_runs.py",
        "--seeds", *map(str, SEEDS),
        "--models", *MODELS,
        "--horizons", *map(str, HORIZONS),
        "--copy"]          # copies, not symlinks, so the merged tree stands alone in Drive

print(subprocess.run(args + ["--dry-run"], cwd=REPO_DIR, capture_output=True, text=True).stdout)
result = subprocess.run(args + ["--force"], cwd=REPO_DIR, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print(result.stderr)

merged_src = pathlib.Path(REPO_DIR) / "results" / "experiments_merged"
if merged_src.is_dir():
    merged_dst = pathlib.Path(f"{DRIVE_RESULTS}/experiments_merged")
    if merged_dst.exists():
        shutil.rmtree(merged_dst)
    shutil.copytree(merged_src, merged_dst)
    print(f"Merged tree copied to {merged_dst}")

## 8 · Status

In [ ]:
import pathlib, yaml

done = set()
for resolved in EXP_DIR.glob("*/resolved_config.yaml"):
    if (resolved.parent / "aggregate_metrics.json").is_file():
        try:
            done.add(yaml.safe_load(resolved.read_text())["experiment"]["name"])
        except Exception:
            pass

print(f"{'cell':<22}" + "".join(f"seed {s:<6}" for s in SEEDS))
print("-" * (22 + 11 * len(SEEDS)))
complete = 0
for model in MODELS:
    for horizon in HORIZONS:
        cell = f"full_{model}_{horizon}min"
        marks = []
        for s in SEEDS:
            ok = f"{cell}_seed{s}" in done
            complete += ok
            marks.append(f"{'done' if ok else '--':<11}")
        print(f"{cell:<22}" + "".join(marks))
print(f"\n{complete}/{len(MODELS)*len(HORIZONS)*len(SEEDS)} cells complete")
merged = pathlib.Path(REPO_DIR) / "results" / "experiments_merged"
print(f"merged cells: {len(list(merged.glob('full_*'))) if merged.is_dir() else 0}")

In [ ]:
import pathlib, shutil, subprocess

EXP_DIR = pathlib.Path(REPO_DIR) / "results" / "experiments"
EXP_DIR.mkdir(parents=True, exist_ok=True)

restored = 0
for d in sorted(pathlib.Path(f"{DRIVE_RESULTS}/experiments").glob("experiment_*")):
    target = EXP_DIR / d.name
    if not target.exists():
        shutil.copytree(d, target)
        restored += 1
# The merged tree too: sections 10-14 read it, and a fresh session that did the
# merging in an earlier one would otherwise find nothing to analyse.
_merged_drive = pathlib.Path(f"{DRIVE_RESULTS}/experiments_merged")
_merged_local = pathlib.Path(REPO_DIR) / "results" / "experiments_merged"
if _merged_drive.is_dir():
    shutil.copytree(_merged_drive, _merged_local, dirs_exist_ok=True)
    print(f"Merged tree restored from {_merged_drive}")

print(f"Restored {restored} experiment dir(s) from Drive.\n")

env = dict(os.environ, DRY_RUN="1",
           SEEDS=" ".join(map(str, SEEDS)),
           MODELS=" ".join(MODELS),
           HORIZONS=" ".join(map(str, HORIZONS)),
           PYTHON=sys.executable)
print(subprocess.run(["./run_seed_major.sh"], cwd=REPO_DIR, env=env,
                     capture_output=True, text=True).stdout)

## 9 · Prepare the experiment analyses

The following cells run every stage of `RUN/run_experiments_analysis.sh`, in
script order, using the merged runs from section 7. Each stage can be rerun
independently; rerunning recomputes its outputs. No training is performed.

After a runtime reset, run sections 1–3 to restore the code and raw data, then
start here: merged runs are restored from Drive when absent locally. If a merged
cell is missing, complete training and section 7 first.

Full resampling counts come from `RUN/_common.sh`. Set `ANALYSIS_QUICK = True`
for a smoke check only; rerun with `False` for publication results. Optional
overrides below use the same environment variables as the shell driver.
Logs go directly to Drive. Shared analysis outputs and each merged run's
`analysis/` folder are copied to Drive after every stage, including on failure.


In [ ]:
import os, pathlib, shutil, subprocess

ANALYSIS_QUICK = False
ANALYSIS_OVERRIDES = {
    # "REPLICATES": "20000",
    # "PERMUTATIONS": "10000",
    # "RAPID_THRESHOLD": "15",
    # "FIGURE_PATIENTS": "all",  # or, e.g., "540 584"
    # "FIGURE_MODE": "transfer",
    # "TEST_FAMILY": "non-parametric",
    # "CORRECTION": "holm",
}

repo = pathlib.Path(REPO_DIR)
for script in ("RUN/_common.sh", "RUN/run_experiments_analysis.sh",
               "RUN/run_dataset_analysis.sh"):
    if not (repo / script).is_file():
        raise FileNotFoundError(f"Missing {script}; update the repository checkout.")

analysis_merged = repo / "results" / "experiments_merged"
analysis_output = repo / "results" / "analysis"
analysis_logs = pathlib.Path(DRIVE_RESULTS) / "logs" / "analysis"
analysis_logs.mkdir(parents=True, exist_ok=True)
analysis_parents = []
missing = []
for model in MODELS:
    for horizon in HORIZONS:
        parent = analysis_merged / f"full_{model}_{horizon}min"
        saved = pathlib.Path(DRIVE_RESULTS) / "experiments_merged" / parent.name
        if not parent.exists() and saved.is_dir():
            shutil.copytree(saved, parent)
        required = ["aggregate_metrics.json", "resolved_config.yaml", "tracking.json"]
        required += [f"{mode}/seed_{seed}/metrics.json"
                     for mode in ("regular", "transfer") for seed in SEEDS]
        absent = [name for name in required if not (parent / name).is_file()]
        if absent:
            missing.append(f"{parent.name}: {', '.join(absent)}")
        else:
            # The shell driver's run-list format requires whitespace-free paths.
            relative = parent.relative_to(repo).as_posix()
            analysis_parents.append(f"{model} {horizon} {relative}\n")
if missing:
    raise RuntimeError("Complete training and rerun section 7 before analysis:\n" +
                       "\n".join(missing))

(analysis_logs / "parents.txt").write_text("".join(analysis_parents))
analysis_env = dict(os.environ, **ANALYSIS_OVERRIDES)
analysis_env.update(
    MODELS=" ".join(MODELS), HORIZONS=" ".join(map(str, HORIZONS)),
    SEEDS=" ".join(map(str, SEEDS)), LOG_DIR=str(analysis_logs),
    ANALYSIS_DIR=str(analysis_output), DATASET_DIR=str(analysis_output / "dataset"),
    DATA_ROOT=str(repo / "data"),
    FIGURE_MODEL=ANALYSIS_OVERRIDES.get("FIGURE_MODEL", "gru" if "gru" in MODELS else MODELS[0]),
    FIGURE_HORIZON=ANALYSIS_OVERRIDES.get("FIGURE_HORIZON", str(30 if 30 in HORIZONS else HORIZONS[0])),
    FIGURE_SEED=ANALYSIS_OVERRIDES.get("FIGURE_SEED", str(SEEDS[0])),
)

def mirror_analysis_to_drive():
    if analysis_output.is_dir():
        shutil.copytree(analysis_output, pathlib.Path(DRIVE_RESULTS) / "analysis",
                        dirs_exist_ok=True)
    for row in analysis_parents:
        _, _, relative = row.split()
        parent = repo / relative
        if (parent / "analysis").is_dir():
            target = pathlib.Path(DRIVE_RESULTS) / "experiments_merged" / parent.name / "analysis"
            shutil.copytree(parent / "analysis", target, dirs_exist_ok=True)

def run_analysis_stage(stage, *, dataset=False, check=False):
    script = "RUN/run_dataset_analysis.sh" if dataset else "RUN/run_experiments_analysis.sh"
    command = ["bash", script, stage]
    if check:
        command.append("--check")
    if ANALYSIS_QUICK and not dataset:
        command.append("--quick")
    try:
        # Stream progress to Colab; the drivers also retain full logs on Drive.
        subprocess.run(command, cwd=repo, env=analysis_env, check=True)
    finally:
        if not check:
            mirror_analysis_to_drive()

print(f"Recorded {len(analysis_parents)} merged runs in {analysis_logs / 'parents.txt'}")
run_analysis_stage("all", check=True)


## 10 · Dataset signal features required by shift analysis

Generate the train/test distribution-shift and signal-irregularity table and
its metadata before running the experiment stages. This runs the `signal`
stage of `RUN/run_dataset_analysis.sh` using the same merged runs and paths.


In [ ]:
run_analysis_stage("signal", dataset=True)


## 11 · Clarke zone-D uncertainty

Nested patient × seed bootstrap of Clarke zone-D errors.


In [ ]:
run_analysis_stage("zone-d")


## 12 · Error by glycemic range

Signed and absolute errors by glycemic band, with nested bootstrap intervals.


In [ ]:
run_analysis_stage("error-range")


## 13 · Error during rapid glucose changes

Calm versus rapid-change errors with nested bootstrap intervals.


In [ ]:
run_analysis_stage("rapid-change")


## 14 · Per-patient error-localization figures

Error-localization plots and pooled-seed Clarke grids for the selected figure mode and patients.


In [ ]:
run_analysis_stage("figures")


## 15 · Per-patient Clarke grids

Single-seed Clarke grids alongside the pooled-seed views.


In [ ]:
run_analysis_stage("glycemic")


## 16 · Glucose-window t-SNE

Cohort-level embedding for the selected figure model, horizon and mode.


In [ ]:
run_analysis_stage("tsne")


## 17 · Persistence baseline

Forecast-origin persistence baseline over every completed prediction run.


In [ ]:
run_analysis_stage("persistence")


## 18 · Shift and patient difficulty

Per-model permutation and bootstrap analyses, including transfer-benefit comparisons when available.


In [ ]:
run_analysis_stage("shift")


## 19 · Transfer versus regular learning

Confirmatory paired inference per model across its horizons.


In [ ]:
run_analysis_stage("transfer")


## 20 · Patient-difficulty stability

Patient-difficulty concordance across models, horizons and learning modes.


In [ ]:
run_analysis_stage("stability")


## 21 · Cross-horizon significance

Cross-horizon tests per model and mode using the configured test family and correction.


In [ ]:
run_analysis_stage("horizons")


## 22 · Cross-model comparison

Compare model performance across the recorded merged runs.


In [ ]:
run_analysis_stage("compare")


## 23 · Analysis output locations

Print the shell driver’s output catalogue. Drive copies live under `DRIVE_RESULTS/analysis`,
`DRIVE_RESULTS/experiments_merged/full_<model>_<horizon>min/analysis`, and
`DRIVE_RESULTS/logs/analysis`.


In [ ]:
run_analysis_stage("summary")
